# Xenium Human Lung Cancer (FFPE) workflow

## Input data: two lung cancer FFPE datasets from 10x Genomics (CC BY 4.0)

| | **Dataset A (start here)** | **Dataset B (scale-up)** |
|---|---|---|
| Platform | Xenium v1, Onboard Analysis 2.0.0, multimodal segmentation | Xenium Prime 5K, Onboard Analysis 3.0.0 |
| Gene panel | Human Multi-Tissue & Cancer (377 genes + 20 negative-control probes) | 5K Pan Tissue & Pathways (~5,100 genes) |
| Tissue area | 30.3 mm² | 71.1 mm² |
| Cells | 162,254 | 278,328 |
| High-quality transcripts (Q20) | 10.6 M | 149.4 M |
| Median transcripts per cell | 46 | 242 |
| Median genes per cell | 29 | 196 |
| Share of transcripts assigned to cells | 86.6% | 82.0% |
| Download size (zip) | ~8.0 GB | ~22.6 GB |

**Download base URLs**
- Dataset A: `https://cf.10xgenomics.com/samples/xenium/2.0.0/Xenium_V1_humanLung_Cancer_FFPE/` ([dataset page](https://www.10xgenomics.com/datasets/preview-data-ffpe-human-lung-cancer-with-xenium-multimodal-cell-segmentation-1-standard))
- Dataset B: `https://cf.10xgenomics.com/samples/xenium/3.0.0/Xenium_Prime_Human_Lung_Cancer_FFPE/` ([dataset page](https://www.10xgenomics.com/datasets/xenium-human-lung-cancer-post-xenium-technote))

### Dataset A's other QC numbers
- 87.4% of transcripts pass the Q20 quality filter.
- The negative-control rate is very low (about 0.05 false-positive transcripts per cell).
- Cell segmentation used three methods: 33% of cells from a boundary stain, 65% from an interior stain, and 1.5% by expanding the nucleus outline.

### Its panel covers what a TIME analysis needs
- **T cells:** CD3D/E, CD8A, CD4, FOXP3, TRAC
- **Checkpoints:** PDCD1, CD274 (PD-L1), CTLA4, LAG3, HAVCR2
- **Myeloid:** CD68, CD163, MARCO, TREM2
- **B and plasma cells:** MS4A1, CD79A, MZB1
- **Tumour and epithelium:** EPCAM, KRT7, EGFR, ERBB2, MET
- **Stroma:** PDGFRA/B, ACTA2
- **Proliferation:** MKI67, TOP2A


### Proposed pipeline

1. Download Dataset A.
2. Input data overview: annotated folder tree and a data card of every file (headers only, via pyarrow, h5py and tifffile).
3. Load it with spatialdata-io into a SpatialData object and explore the raw tables (transcripts, cells, boundaries) with pyarrow.
4. Run QC.
5. Normalise and cluster.
6. Annotate cell types.
7. Spatial analysis (neighbourhood enrichment, niches, immune infiltration).
8. Align the H&E image.
9. Optionally, repeat on Dataset B.

## Step 1: Download Dataset A

Raw data goes to `~/data/xenium_lung/`, **outside OneDrive**, so multi-GB files don't clog sync.
The cell is safe to re-run: finished files are skipped and interrupted downloads resume where they stopped.

- `DOWNLOAD_HE = True` also fetches the ~1.2 GB matched H&E image, used in Step 8 to map cells onto pathology regions (tumour, stroma, TLS). Set it to `False` only if disk space is tight; re-running later fetches just the missing image.
- Needs ~8 GB for the zip plus roughly the same again once unzipped; set `DELETE_ZIP_AFTER_UNZIP = True` to reclaim the zip space.


In [1]:
import shutil
import zipfile
from pathlib import Path

import requests
from tqdm.auto import tqdm

# ---- Settings ---------------------------------------------------------------
DATA_ROOT = Path("~/data/xenium_lung").expanduser()   # outside OneDrive
DOWNLOAD_HE = True              # ~1.2 GB matched H&E image for Step 8 alignment
DELETE_ZIP_AFTER_UNZIP = False  # set True to free ~8 GB once extracted

SAMPLE = "Xenium_V1_humanLung_Cancer_FFPE"
BASE_URL = f"https://cf.10xgenomics.com/samples/xenium/2.0.0/{SAMPLE}"

FILES = [
    f"{SAMPLE}_metrics_summary.csv",
    f"{SAMPLE}_gene_panel.json",
    f"{SAMPLE}_he_imagealignment.csv",
    f"{SAMPLE}_outs.zip",
]
if DOWNLOAD_HE:
    FILES.append(f"{SAMPLE}_he_image.ome.tif")

DATA_DIR = DATA_ROOT / SAMPLE
# ask for raw bytes so byte counts match on-disk sizes (no gzip transfer encoding)
NO_GZIP = {"Accept-Encoding": "identity"}
DATA_DIR.mkdir(parents=True, exist_ok=True)


def remote_size(url):
    r = requests.head(url, headers=NO_GZIP, allow_redirects=True, timeout=60)
    r.raise_for_status()
    return int(r.headers.get("content-length", 0))


def download(url, dest):
    """Resumable download: skips complete files, continues partial ones."""
    total = remote_size(url)
    have = dest.stat().st_size if dest.exists() else 0
    if total and have == total:
        print(f"[skip] {dest.name} already complete ({total / 1e9:.2f} GB)")
        return
    if have > total:  # corrupt / mismatched partial, start over
        dest.unlink()
        have = 0

    headers = {**NO_GZIP, **({"Range": f"bytes={have}-"} if have else {})}
    with requests.get(url, headers=headers, stream=True, timeout=60) as r:
        r.raise_for_status()
        if have and r.status_code != 206:  # server ignored Range, restart
            have = 0
        mode = "ab" if have else "wb"
        with open(dest, mode) as f, tqdm(
            total=total, initial=have, unit="B", unit_scale=True, desc=dest.name
        ) as bar:
            for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                f.write(chunk)
                bar.update(len(chunk))

    if dest.stat().st_size != total:
        raise IOError(f"{dest.name}: size mismatch, re-run this cell to resume")


# ---- Disk space check -------------------------------------------------------
needed = sum(remote_size(f"{BASE_URL}/{f}") for f in FILES)
free = shutil.disk_usage(DATA_ROOT).free
print(f"Download size: {needed / 1e9:.1f} GB | free disk: {free / 1e9:.1f} GB")
if free < 2.2 * needed:  # zip + extracted copy + headroom
    print("WARNING: disk space is tight for download + unzip.")

# ---- Download ---------------------------------------------------------------
for fname in FILES:
    download(f"{BASE_URL}/{fname}", DATA_DIR / fname)

# ---- Unzip the output bundle -------------------------------------------------
zip_path = DATA_DIR / f"{SAMPLE}_outs.zip"
OUTS_DIR = DATA_DIR / "outs"
if not (OUTS_DIR / "experiment.xenium").exists():
    with zipfile.ZipFile(zip_path) as zf:
        members = zf.infolist()
        # strip a single top-level folder (e.g. "outs/") if the zip has one
        tops = {m.filename.split("/")[0] for m in members}
        prefix = tops.pop() + "/" if len(tops) == 1 else ""
        OUTS_DIR.mkdir(exist_ok=True)
        for m in tqdm(members, desc="unzip"):
            rel = m.filename[len(prefix):]
            if not rel:
                continue
            target = OUTS_DIR / rel
            if m.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(m) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, length=8 * 1024 * 1024)
    if DELETE_ZIP_AFTER_UNZIP:
        zip_path.unlink()
else:
    print("[skip] outs/ already extracted")

# ---- Inventory (relative paths only) -----------------------------------------
print(f"\nContents of {SAMPLE}/outs:")
for item in sorted(OUTS_DIR.iterdir()):
    if item.is_file():
        size = item.stat().st_size
    else:
        size = sum(f.stat().st_size for f in item.rglob("*") if f.is_file())
    print(f"  {item.name:<40} {size / 1e6:>10.1f} MB")


Download size: 9.2 GB | free disk: 47.4 GB


Xenium_V1_humanLung_Cancer_FFPE_metrics_summary.csv:   0%|          | 0.00/1.73k [00:00<?, ?B/s]

Xenium_V1_humanLung_Cancer_FFPE_gene_panel.json:   0%|          | 0.00/209k [00:00<?, ?B/s]

Xenium_V1_humanLung_Cancer_FFPE_he_imagealignment.csv:   0%|          | 0.00/126 [00:00<?, ?B/s]

Xenium_V1_humanLung_Cancer_FFPE_outs.zip:   0%|          | 0.00/7.99G [00:00<?, ?B/s]

Xenium_V1_humanLung_Cancer_FFPE_he_image.ome.tif:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

unzip:   0%|          | 0/26 [00:00<?, ?it/s]


Contents of Xenium_V1_humanLung_Cancer_FFPE/outs:
  analysis.tar.gz                                32.1 MB
  analysis.zarr.zip                               1.4 MB
  analysis_summary.html                          10.4 MB
  aux_outputs.tar.gz                             50.2 MB
  cell_boundaries.csv.gz                         38.4 MB
  cell_boundaries.parquet                        14.4 MB
  cell_feature_matrix.h5                          6.7 MB
  cell_feature_matrix.tar.gz                     23.0 MB
  cell_feature_matrix.zarr.zip                    9.1 MB
  cells.csv.gz                                    8.8 MB
  cells.parquet                                   2.9 MB
  cells.zarr.zip                                147.1 MB
  experiment.xenium                               0.0 MB
  gene_panel.json                                 0.2 MB
  metrics_summary.csv                             0.0 MB
  morphology.ome.tif                           4762.9 MB
  morphology_focus                   

## Step 2: Input data overview

Before any analysis, get to know the input data. This step answers two questions, and **neither loads the data into memory** (it only reads file headers and metadata, so it runs in about a second):

1. **What files do I have?** (2a) A tree of every file and folder with its size, a role tag, and a plain-language description. Compressed archives are listed without being extracted.
2. **What is inside each file?** (2b) A data card: run metadata, table columns with their meaning and a 3-row preview, the count-matrix layout, image dimensions and stains, and the H&E alignment.

| Role tag | Meaning |
|---|---|
| `CORE` | Primary analysis inputs you will load (transcripts, cells, boundaries, count matrix) |
| `IMAGE` | Microscopy images (DAPI z-stack, segmentation stains, H&E) |
| `META` | Run and panel metadata |
| `QC` | Quality reports and technical extras |
| `BASELINE` | 10x's own automated clustering, to benchmark your results against |
| `EXPLORER` | Xenium Explorer viewer formats (not needed for Python analysis) |
| `COPY` | Same content as a `CORE` file in another format (CSV, MEX) |
| `ARCHIVE` | Compressed download bundle |

### 2a. Data folder tree


In [2]:
import tarfile
from collections import defaultdict
from pathlib import Path

# Re-declared here so this cell runs on its own after a kernel restart
DATA_ROOT = Path("~/data/xenium_lung").expanduser()
SAMPLE = "Xenium_V1_humanLung_Cancer_FFPE"
SAMPLE_DIR = DATA_ROOT / SAMPLE
OUTS_DIR = SAMPLE_DIR / "outs"

# Role tags tell you at a glance how each file is used in the workflow
#   CORE     primary analysis input (you will load these)
#   IMAGE    microscopy images
#   META     run / panel metadata
#   QC       quality reports
#   BASELINE 10x's own automated analysis, to benchmark your results against
#   EXPLORER Xenium Explorer viewer format (not needed for Python analysis)
#   COPY     same content as a CORE file, different format
#   ARCHIVE  compressed bundle
P = SAMPLE + "_"
FILE_INFO = {
    # ---- top level: files downloaded individually from 10x ----
    f"{P}outs.zip":                 ("ARCHIVE",  "Original 10x download of outs/. Safe to delete once unzipped"),
    f"{P}he_image.ome.tif":         ("IMAGE",    "H&E stain of the same section (RGB, pyramidal). For pathology context"),
    f"{P}he_imagealignment.csv":    ("META",     "3x3 affine matrix: H&E pixels -> Xenium morphology pixels"),
    f"{P}gene_panel.json":          ("META",     "Panel definition (copy of outs/gene_panel.json)"),
    f"{P}metrics_summary.csv":      ("QC",       "Run QC numbers (copy of outs/metrics_summary.csv)"),
    # ---- outs/: the Xenium Onboard Analysis bundle ----
    "outs":                         ("",         "Xenium Onboard Analysis 2.0 output bundle"),
    "outs/experiment.xenium":       ("META",     "Run manifest (JSON): panel, pixel size, file pointers. Readers open this"),
    "outs/gene_panel.json":         ("META",     "Panel: 377 genes + negative controls, probe and codeword details"),
    "outs/metrics_summary.csv":     ("QC",       "One-row QC table: cells, transcripts, Q20 rate, negative-control rate"),
    "outs/analysis_summary.html":   ("QC",       "10x web report (QC plots + clustering). Open in a browser first"),
    "outs/transcripts.parquet":     ("CORE",     "Every decoded transcript: gene, x/y/z (um), quality qv, assigned cell_id"),
    "outs/transcripts.csv.gz":      ("COPY",     "Same as transcripts.parquet, as CSV"),
    "outs/transcripts.zarr.zip":    ("EXPLORER", "Transcripts for the Xenium Explorer viewer"),
    "outs/cells.parquet":           ("CORE",     "One row per cell: centroid x/y (um), cell/nucleus area, counts"),
    "outs/cells.csv.gz":            ("COPY",     "Same as cells.parquet, as CSV"),
    "outs/cells.zarr.zip":          ("EXPLORER", "Cell + nucleus masks and summaries for Xenium Explorer"),
    "outs/cell_boundaries.parquet": ("CORE",     "Cell outline polygons: vertex x/y (um) per cell_id"),
    "outs/cell_boundaries.csv.gz":  ("COPY",     "Same as cell_boundaries.parquet, as CSV"),
    "outs/nucleus_boundaries.parquet": ("CORE",  "Nucleus outline polygons: vertex x/y (um) per cell_id"),
    "outs/nucleus_boundaries.csv.gz":  ("COPY",  "Same as nucleus_boundaries.parquet, as CSV"),
    "outs/cell_feature_matrix.h5":  ("CORE",     "Cell x feature count matrix (genes + controls). Loads into AnnData"),
    "outs/cell_feature_matrix.tar.gz": ("COPY",  "Same matrix in MEX format (matrix.mtx, barcodes, features)"),
    "outs/cell_feature_matrix.zarr.zip": ("EXPLORER", "Count matrix for Xenium Explorer"),
    "outs/morphology.ome.tif":      ("IMAGE",    "3D DAPI (nuclei) z-stack, full resolution, pyramidal"),
    "outs/morphology_focus":        ("IMAGE",    "2D in-focus images, one stain per file (used for segmentation)"),
    "outs/analysis.tar.gz":         ("BASELINE", "10x clustering, differential expression, PCA, UMAP, t-SNE (CSV)"),
    "outs/analysis.zarr.zip":       ("EXPLORER", "10x clustering results for Xenium Explorer"),
    "outs/aux_outputs.tar.gz":      ("QC",       "Technical extras: per-cycle raw channel images, FOV tile positions"),
}

# Descriptions for paths INSIDE tar.gz archives (shown without extracting)
ARCHIVE_INFO = {
    "analysis":            "Archive contents (listed, not extracted)",
    "cell_feature_matrix": "Archive contents (listed, not extracted)",
    "aux_outputs":         "Archive contents (listed, not extracted)",
    "analysis/clustering":       "Cluster label per cell (graph-based + k-means k=2..10)",
    "analysis/diffexp":          "Marker genes per cluster (log2 fold change, p-values)",
    "analysis/pca":              "PCA: loadings, per-cell projection, variance explained",
    "analysis/umap":             "2D UMAP coordinates per cell",
    "analysis/tsne":             "2D t-SNE coordinates per cell",
    "cell_feature_matrix/barcodes.tsv.gz": "Cell IDs (matrix columns)",
    "cell_feature_matrix/features.tsv.gz": "Feature IDs, names and types (matrix rows)",
    "cell_feature_matrix/matrix.mtx.gz":   "Sparse counts (Matrix Market)",
    "aux_outputs/per_cycle_channel_images": "Raw image per imaging cycle x color channel",
    "aux_outputs/morphology_fov_locations.json": "Position of each field of view (FOV) tile",
}
PEEK_INSIDE = {"outs/analysis.tar.gz", "outs/cell_feature_matrix.tar.gz", "outs/aux_outputs.tar.gz"}
MAX_CHILDREN = 6  # inside archives, collapse long lists of similar entries


def human(n):
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1000 or unit == "TB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1000


def path_size(p):
    return p.stat().st_size if p.is_file() else sum(f.stat().st_size for f in p.rglob("*") if f.is_file())


def focus_channels(focus_dir):
    """Map each morphology_focus file to its stain, read from the OME-XML header."""
    try:
        import re
        import tifffile
        first = sorted(focus_dir.glob("*.ome.tif"))[0]
        with tifffile.TiffFile(first) as t:
            xml = t.ome_metadata or ""
        names = re.findall(r'<Channel [^>]*Name="([^"]+)"', xml)
        files = re.findall(r'<TiffData FirstC="(\d+)"[^>]*>\s*<UUID[^>]*FileName="([^"]+)"', xml)
        return {fname: names[int(c)] for c, fname in files if int(c) < len(names)}
    except Exception:
        return {}


STAIN_ROLE = {
    "DAPI": "nuclei",
    "18S": "ribosomal RNA, cell interior",
    "ATP1A1/CD45/E-Cadherin": "membrane, cell boundary",
    "alphaSMA/Vimentin": "cytoskeleton, cell interior",
}

rows = []  # (tree_prefix + name, size_str, role, description)
role_bytes = defaultdict(int)


def add(prefix, name, size, role="", desc=""):
    rows.append((prefix + name, human(size) if size is not None else "", role, desc))


def archive_tree(tar_path, prefix):
    """List a tar.gz as a tree (sizes = uncompressed) without extracting it."""
    tree = {}
    with tarfile.open(tar_path, "r:gz") as tf:
        for m in tf.getmembers():
            node = tree
            parts = m.name.strip("/").split("/")
            for part in parts[:-1]:
                node = node.setdefault(part, {"__size__": 0})
                node["__size__"] += m.size
            leaf = node.setdefault(parts[-1], {"__size__": 0})
            if m.isfile():
                leaf["__size__"] = m.size

    def walk(node, path, pre):
        kids = [k for k in node if k != "__size__"]
        shown = kids if len(kids) <= MAX_CHILDREN + 1 else kids[:MAX_CHILDREN]
        for i, k in enumerate(shown):
            last = i == len(shown) - 1 and len(shown) == len(kids)
            child, cpath = node[k], f"{path}/{k}" if path else k
            is_dir = any(x != "__size__" for x in child)
            add(pre + ("└── " if last else "├── "), k + ("/" if is_dir else ""),
                child["__size__"], "", ARCHIVE_INFO.get(cpath, ""))
            # always open the archive's top folder; stop at any described sub-folder
            if is_dir and ("/" not in cpath or cpath not in ARCHIVE_INFO):
                walk(child, cpath, pre + ("    " if last else "│   "))
        if len(shown) < len(kids):
            add(pre + "└── ", f"... +{len(kids) - len(shown)} more", None)

    walk(tree, "", prefix)


def walk_dir(directory, prefix=""):
    entries = sorted(directory.iterdir(), key=lambda p: (p.is_dir(), p.name.lower()))
    entries = [e for e in entries if not e.name.startswith(".")]
    for i, entry in enumerate(entries):
        last = i == len(entries) - 1
        rel = entry.relative_to(SAMPLE_DIR).as_posix()
        role, desc = FILE_INFO.get(rel, ("", ""))
        size = path_size(entry)
        if entry.is_file() and role:
            role_bytes[role] += size
        add(prefix + ("└── " if last else "├── "), entry.name + ("/" if entry.is_dir() else ""), size, role, desc)
        child_prefix = prefix + ("    " if last else "│   ")
        if entry.is_dir():
            if entry.name == "morphology_focus":
                stains = focus_channels(entry)
                files = sorted(entry.iterdir())
                for j, f in enumerate(files):
                    stain = stains.get(f.name, "?")
                    role_bytes["IMAGE"] += f.stat().st_size
                    add(child_prefix + ("└── " if j == len(files) - 1 else "├── "), f.name,
                        f.stat().st_size, "IMAGE", f"Stain: {stain} ({STAIN_ROLE.get(stain, '')})")
            else:
                walk_dir(entry, child_prefix)
        elif rel in PEEK_INSIDE:
            archive_tree(entry, child_prefix)


add("", SAMPLE + "/", path_size(SAMPLE_DIR), "", "Dataset A: human lung cancer, FFPE, Xenium v1 (377 genes)")
walk_dir(SAMPLE_DIR)

# ---- print as an aligned table ----------------------------------------------
w_name = max(len(r[0]) for r in rows)
w_size = max(len(r[1]) for r in rows)
w_role = max(len(r[2]) for r in rows) + 3
print(f"{'PATH':<{w_name}}  {'SIZE':>{w_size}}  {'ROLE':<{w_role}}DESCRIPTION")
print("-" * (w_name + w_size + w_role + 60))
for name, size, role, desc in rows:
    tag = f"[{role}]" if role else ""
    print(f"{name:<{w_name}}  {size:>{w_size}}  {tag:<{w_role}}{desc}")

# ---- disk usage by role -------------------------------------------------------
print("\nDisk usage by role (top-level files and outs/; archive contents not double-counted)")
total = sum(role_bytes.values())
for role, b in sorted(role_bytes.items(), key=lambda kv: -kv[1]):
    print(f"  {role:<9} {human(b):>9}  {b / total:6.1%}")
print(f"  {'TOTAL':<9} {human(total):>9}")
print("\nSizes are decimal (1 GB = 10^9 bytes). Archive contents show uncompressed sizes.")

PATH                                                           SIZE  ROLE       DESCRIPTION
----------------------------------------------------------------------------------------------------------------------------------------
Xenium_V1_humanLung_Cancer_FFPE/                            17.2 GB             Dataset A: human lung cancer, FFPE, Xenium v1 (377 genes)
├── Xenium_V1_humanLung_Cancer_FFPE_gene_panel.json        209.1 KB  [META]     Panel definition (copy of outs/gene_panel.json)
├── Xenium_V1_humanLung_Cancer_FFPE_he_image.ome.tif         1.2 GB  [IMAGE]    H&E stain of the same section (RGB, pyramidal). For pathology context
├── Xenium_V1_humanLung_Cancer_FFPE_he_imagealignment.csv     126 B  [META]     3x3 affine matrix: H&E pixels -> Xenium morphology pixels
├── Xenium_V1_humanLung_Cancer_FFPE_metrics_summary.csv      1.7 KB  [QC]       Run QC numbers (copy of outs/metrics_summary.csv)
├── Xenium_V1_humanLung_Cancer_FFPE_outs.zip                 8.0 GB  [ARCHIVE]  Origina

### 2b. Data card: what is inside each file

Key points to notice in the output:
- **Coordinates are in microns (µm)** in every table; image pixels are 0.2125 µm. Divide µm by 0.2125 to get morphology pixel positions.
- **`cell_id` is the join key** across transcripts, cells, boundaries and the count matrix.
- **Not every transcript is a gene**: the matrix has 541 features, of which 377 are genes and 164 are background controls. Filter to `Gene Expression` and `qv >= 20` before biology.
- **Boundary tables have one row per polygon vertex**, not per cell.
- **The H&E is rotated about 90° and scaled** relative to Xenium; the alignment matrix corrects both.


In [3]:
import json
import re
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import tifffile
from IPython.display import display

DATA_ROOT = Path("~/data/xenium_lung").expanduser()
SAMPLE = "Xenium_V1_humanLung_Cancer_FFPE"
SAMPLE_DIR = DATA_ROOT / SAMPLE
OUTS_DIR = SAMPLE_DIR / "outs"

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)


def header(title):
    print(f"\n{'=' * 90}\n{title}\n{'=' * 90}")


# ---- 1. Run metadata ---------------------------------------------------------
header("1. RUN METADATA  (outs/experiment.xenium)")
exp = json.loads((OUTS_DIR / "experiment.xenium").read_text())
keys = ["run_name", "region_name", "preservation_method", "panel_name", "panel_design_id",
        "panel_num_targets_predesigned", "num_cells", "transcripts_per_cell",
        "pixel_size", "z_step_size", "analysis_sw_version", "segmentation_stain", "run_start_time"]
display(pd.DataFrame({"value": [exp.get(k) for k in keys]}, index=keys))

# ---- 2. Tables: schema + preview (metadata only, nothing large is loaded) -----
COLUMN_INFO = {
    "transcripts.parquet": {
        "transcript_id": "Unique ID of the detected transcript",
        "cell_id": "Cell it was assigned to ('UNASSIGNED' = outside any cell)",
        "overlaps_nucleus": "1 if the transcript lies inside the nucleus",
        "feature_name": "Gene name, or control probe / codeword name",
        "x_location": "X position (um)",
        "y_location": "Y position (um)",
        "z_location": "Z depth in the section (um)",
        "qv": "Decoding quality (Phred-like). Keep qv >= 20",
        "fov_name": "Imaging field-of-view tile it came from",
        "nucleus_distance": "Distance to the nearest nucleus edge (um)",
        "codeword_index": "Index of the optical barcode (codeword)",
    },
    "cells.parquet": {
        "cell_id": "Unique cell ID (joins to every other table)",
        "x_centroid": "Cell centre X (um)",
        "y_centroid": "Cell centre Y (um)",
        "transcript_counts": "High-quality gene transcripts in the cell",
        "control_probe_counts": "Negative-control probe counts (background)",
        "control_codeword_counts": "Negative-control codeword counts (decoding errors)",
        "unassigned_codeword_counts": "Unused-codeword counts (decoding errors)",
        "deprecated_codeword_counts": "Deprecated-codeword counts",
        "total_counts": "All counts in the cell, controls included",
        "cell_area": "Cell area (um^2)",
        "nucleus_area": "Nucleus area (um^2)",
    },
    "cell_boundaries.parquet": {
        "cell_id": "Cell the vertex belongs to",
        "vertex_x": "Polygon vertex X (um)",
        "vertex_y": "Polygon vertex Y (um)",
    },
    "nucleus_boundaries.parquet": {
        "cell_id": "Cell whose nucleus the vertex belongs to",
        "vertex_x": "Polygon vertex X (um)",
        "vertex_y": "Polygon vertex Y (um)",
    },
}

for fname, col_info in COLUMN_INFO.items():
    pf = pq.ParquetFile(OUTS_DIR / fname)
    schema = pf.schema_arrow
    header(f"2. TABLE  outs/{fname}   rows: {pf.metadata.num_rows:,}   columns: {len(schema)}")
    display(pd.DataFrame({
        "column": schema.names,
        "dtype": [str(t) for t in schema.types],
        "meaning": [col_info.get(c, "") for c in schema.names],
    }))
    preview = next(pf.iter_batches(batch_size=3)).to_pandas()
    print("First 3 rows:")
    display(preview)

# ---- 3. Count matrix ------------------------------------------------------------
header("3. COUNT MATRIX  (outs/cell_feature_matrix.h5)")
with h5py.File(OUTS_DIR / "cell_feature_matrix.h5", "r") as h5:
    n_features, n_cells = h5["matrix/shape"][:]
    nnz = h5["matrix/data"].shape[0]
    ftypes = pd.Series([t.decode() for t in h5["matrix/features/feature_type"][:]])
    names = pd.Series([t.decode() for t in h5["matrix/features/name"][:]])
print(f"Shape: {n_cells:,} cells x {n_features:,} features (stored features x cells, CSC sparse)")
print(f"Non-zero entries: {nnz:,}  ({nnz / (n_cells * n_features):.1%} of the matrix is non-zero)")
summary = (pd.DataFrame({"feature_type": ftypes, "name": names})
           .groupby("feature_type")["name"]
           .agg(n="size", examples=lambda s: ", ".join(s.head(4))))
display(summary.sort_values("n", ascending=False))
print("Only 'Gene Expression' features are used for biology; the rest are background controls.")

# ---- 4. Images --------------------------------------------------------------------
header("4. IMAGES  (shape, channels, pyramid levels, physical size)")
images = {
    "outs/morphology.ome.tif": OUTS_DIR / "morphology.ome.tif",
    "outs/morphology_focus/ (4 files, 1 stain each)": OUTS_DIR / "morphology_focus" / "morphology_focus_0000.ome.tif",
    "H&E image": SAMPLE_DIR / f"{SAMPLE}_he_image.ome.tif",
}
img_rows = []
for label, path in images.items():
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        xml = tif.ome_metadata or ""
    channels = re.findall(r'<Channel [^>]*Name="([^"]+)"', xml)
    px = re.search(r'PhysicalSizeX="([\d.]+)"', xml)
    px = float(px.group(1)) if px else None
    axes = series.axes
    h, w = series.shape[axes.index("Y")], series.shape[axes.index("X")]
    img_rows.append({
        "image": label,
        "axes": axes,
        "shape": " x ".join(f"{s:,}" for s in series.shape),
        "dtype": str(series.dtype),
        "pyramid_levels": len(series.levels),
        "um_per_px": px if px else "see alignment",
        "extent_mm (W x H)": f"{w * px / 1000:.1f} x {h * px / 1000:.1f}" if px else "",
        "channels": ", ".join(channels) if channels else "RGB",
    })
display(pd.DataFrame(img_rows).set_index("image"))
print("Axes: Z = focal plane, C = stain channel, Y/X = pixel rows/cols, S = RGB samples.")

# ---- 5. H&E -> Xenium alignment -----------------------------------------------------
header("5. H&E ALIGNMENT  (he_imagealignment.csv)")
M = np.loadtxt(SAMPLE_DIR / f"{SAMPLE}_he_imagealignment.csv", delimiter=",")
scale = np.sqrt(abs(np.linalg.det(M[:2, :2])))
angle = np.degrees(np.arctan2(M[1, 0], M[0, 0]))
print("Affine matrix (H&E pixel -> Xenium morphology pixel):")
print(np.array2string(M, precision=4, suppress_small=True))
print(f"Scale:       {scale:.4f} Xenium px per H&E px  "
      f"(H&E ~{exp['pixel_size'] * scale:.4f} um/px)")
print(f"Rotation:    {angle:.1f} degrees  (the H&E was scanned turned relative to Xenium)")
print(f"Translation: x = {M[0, 2]:,.1f} px, y = {M[1, 2]:,.1f} px")


1. RUN METADATA  (outs/experiment.xenium)


,value
run_name,Human Multi-Tissue Cancer (FFPE)
region_name,Human_Lung_Cancer
preservation_method,FFPE
panel_name,Xenium Human Multi-Tissue and Cancer Gene Expression
panel_design_id,hMulti_v1
panel_num_targets_predesigned,377
num_cells,162254
transcripts_per_cell,46
pixel_size,0.2125
z_step_size,3.0



2. TABLE  outs/transcripts.parquet   rows: 12,165,021   columns: 11


,column,dtype,meaning
0,transcript_id,uint64,Unique ID of the detected transcript
1,cell_id,string,Cell it was assigned to ('UNASSIGNED' = outside any cell)
2,overlaps_nucleus,uint8,1 if the transcript lies inside the nucleus
3,feature_name,string,"Gene name, or control probe / codeword name"
4,x_location,float,X position (um)
5,y_location,float,Y position (um)
6,z_location,float,Z depth in the section (um)
7,qv,float,Decoding quality (Phred-like). Keep qv >= 20
8,fov_name,string,Imaging field-of-view tile it came from
9,nucleus_distance,float,Distance to the nearest nucleus edge (um)


First 3 rows:


...


3. COUNT MATRIX  (outs/cell_feature_matrix.h5)
Shape: 162,254 cells x 541 features (stored features x cells, CSC sparse)
Non-zero entries: 4,901,617  (5.6% of the matrix is non-zero)


,n,examples
feature_type,,
Gene Expression,377,"ABCC11, ACE2, ACKR1, ACTA2"
Unassigned Codeword,103,"UnassignedCodeword_0006, UnassignedCodeword_0037, UnassignedCodeword_0069, U..."
Negative Control Codeword,41,"NegControlCodeword_0500, NegControlCodeword_0501, NegControlCodeword_0502, N..."
Negative Control Probe,20,"NegControlProbe_00042, NegControlProbe_00041, NegControlProbe_00039, NegCont..."


Only 'Gene Expression' features are used for biology; the rest are background controls.

4. IMAGES  (shape, channels, pyramid levels, physical size)


,axes,shape,dtype,pyramid_levels,um_per_px,extent_mm (W x H),channels
image,,,,,,,
outs/morphology.ome.tif,ZYX,"11 x 17,098 x 51,187",uint16,8,0.212500,10.9 x 3.6,DAPI
"outs/morphology_focus/ (4 files, 1 stain each)",CYX,"4 x 17,098 x 51,187",uint16,8,0.212500,10.9 x 3.6,"DAPI, 18S, ATP1A1/CD45/E-Cadherin, alphaSMA/Vimentin"
H&E image,YXS,"45,087 x 11,580 x 3",uint8,6,0.273771,3.2 x 12.3,RGB


Axes: Z = focal plane, C = stain channel, Y/X = pixel rows/cols, S = RGB samples.

5. H&E ALIGNMENT  (he_imagealignment.csv)
Affine matrix (H&E pixel -> Xenium morphology pixel):
[[    0.0048     1.2887   -75.482 ]
 [   -1.2887     0.0048 16309.1028]
 [    0.         0.         1.    ]]
Scale:       1.2887 Xenium px per H&E px  (H&E ~0.2739 um/px)
Rotation:    -89.8 degrees  (the H&E was scanned turned relative to Xenium)
Translation: x = -75.5 px, y = 16,309.1 px
